# Big Data Platforms — Lecture 8 Notes

Course: DATA140031 (MOOC, 3 ECTS) + DATA140032 (MOOC EXAM, 2 ECTS)
Lecturer: Keijo Heljanko, Department of Computer Science, University of Helsinki
Date: 24.9.2026

Lecture 7 introduced the "extensible record store" family of NoSQL databases as
one branch of a four-way taxonomy, with a promise of "more details later."
Lecture 8 is that promised deep dive: **Google BigTable**, the system that
started the whole family, and its open-source clone **Apache HBase**. Almost
every idea from earlier lectures shows up here put to concrete use — lecture 3's
"sequential writes are cheap, random writes are expensive," lecture 6's Chubby
and ZooKeeper as the coordination backbone, and lecture 7's Bloom filters,
here doing real work speeding up reads. The lecture closes by tracing BigTable's
core design idea — the log-structured merge tree — forward into the systems that
still use it today: LevelDB, RocksDB, MyRocks, CockroachDB, and Cassandra.


## 1. BigTable — the paper and the core design bet

BigTable was described in Chang, Dean, Ghemawat, Hsieh, Wallach, Burrows,
Chandra, Fikes, and Gruber, *Bigtable: A Distributed Storage System for
Structured Data*, ACM Trans. Comput. Syst. 26(2) (2008) — one of the most
influential systems papers in this whole course, and the direct ancestor of the
entire "extensible record store" family from lecture 7.

The essentials, up front:

- A **highly scalable, consistent, and partition-tolerant** datastore — a **CP**
  system, in lecture 6's CAP framing.
- Built on top of the **Google File System (GFS)** — Google's internal
  precursor to HDFS, sharing HDFS's core trait from lecture 3: GFS provides
  durability through replication, but only supports **sequential writes**.
- The consequence that shapes everything else in this lecture: **BigTable's
  design does not do random writes at all — only sequential writes are ever
  used.**
- The open-source clone is **Apache HBase**, which gets its own section later.

### Why "only sequential writes" is the whole design, not an incidental detail

Recall from lecture 3: sequential writes are dramatically faster than random
writes, on both hard disks and — for related but different reasons — on flash.
BigTable takes that fact and builds an entire storage engine around never doing
the slow thing. It's explicitly a **write-optimized design**: writes are
optimized at the direct expense of reads. Random reads that miss the DRAM cache
end up slower than they would be in a traditional RDBMS — that's a real,
accepted cost, not an oversight. The design also works hard to **minimize the
total number of bytes written**, through efficient logging and compression —
which, as lecture 5 pointed out, is also exactly the right strategy for
extending the working life of Flash SSDs.


## 2. The BigTable data model

BigTable's own paper describes the system in one memorable phrase: **"a sparse,
distributed, persistent multi-dimensional sorted map."** Every piece of data is
a string, addressed by three coordinates:

- **row** — an arbitrary string row key. Typically 10–100 bytes, with a hard
  maximum of 64 KB.
- **column** — a column key, itself made of a **column family** and a
  **column qualifier** (name), written as `family:qualifier`.
- **timestamp** — a 64-bit integer, commonly used to hold an actual timestamp.

Put together, BigTable is a map with three coordinates going in and a single
string coming out:

```
(row: string, column: string, time: int64) -> string
```


In [1]:
# A minimal illustration of the BigTable data model as a plain Python
# dictionary keyed on the three coordinates. Not how BigTable actually stores
# data on disk (see the SSTable/memtable sections below for that) -- this is
# purely to make the (row, column, time) -> string map concrete.

bigtable_like_map = {}

def put(row, column, timestamp, value):
    bigtable_like_map[(row, column, timestamp)] = value

def get(row, column, timestamp=None):
    if timestamp is not None:
        return bigtable_like_map.get((row, column, timestamp))
    # Without a specific timestamp, BigTable returns the LATEST version --
    # emulate that by picking the max timestamp among matching (row, column)
    matches = [(t, v) for (r, c, t), v in bigtable_like_map.items()
               if r == row and c == column]
    if not matches:
        return None
    matches.sort(key=lambda tv: tv[0], reverse=True)
    return matches[0][1]

put("com.cnn.www", "contents:", 1000, "<html>...v1...</html>")
put("com.cnn.www", "contents:", 2000, "<html>...v2 (updated)...</html>")
put("com.cnn.www", "anchor:cnnsi.com", 1000, "CNN")

print("Latest contents:", get("com.cnn.www", "contents:"))
print("Contents at t=1000:", get("com.cnn.www", "contents:", timestamp=1000))
print("Anchor text:", get("com.cnn.www", "anchor:cnnsi.com"))


Latest contents: <html>...v2 (updated)...</html>
Contents at t=1000: <html>...v1...</html>
Anchor text: CNN


## 3. BigTable data layout — why row keys and column families matter

- Data is stored **sorted by row key**, and automatically **sharded** into
  large blocks that are also sorted by row key. This is what makes efficient
  **range scans** in increasing row-key order possible — asking for
  "everything between row key X and row key Y" is a cheap, sequential
  operation, not a scattered search.
- Columns are grouped into **column families**, and every column belonging to
  one family is stored together, compressed, on disk.
- The direct payoff: a query that only needs columns from *some* families can
  **skip reading the other column families entirely** — the storage layout
  itself makes that skip cheap, since those families' data isn't interleaved
  with the ones being read.
- The **timestamp** coordinate gives applications a natural way to store
  multiple versions of the same logical data side by side — the lecture's
  example is a website whose content changes over time, with each version
  simply living at its own timestamp under the same row and column.


## 4. Worked example — the Web Table

The paper's own running example, and the one the lecture uses to make row keys
and column families concrete: a table storing crawled web pages.

- **Row key: the reverse of the page's URL** — e.g. `com.cnn.www` rather than
  `www.cnn.com`. Reversing the URL groups every page from the same site
  together under adjacent row keys, since they now share a common *prefix*
  rather than a common *suffix* — and adjacent row keys is exactly what makes
  a range scan cheap.
- **`contents:` column family** — stores the web page's actual content, as it
  existed at the timestamp in question.
- **`anchor:` column family** — stores the incoming links pointing *to* this
  page, one column per distinct linking page (so a heavily-linked page can
  genuinely have thousands of columns in this family alone).

The payoff described above becomes concrete here: a query only interested in
the link graph between pages — "what links to what" — never needs to touch the
`contents:` family at all, and the column-family-grouped storage layout makes
skipping it essentially free.


### BigTable Web Table example

![BigTable Web Table example](images/bigtable_webtable.png)



## 5. Atomicity guarantees — row-level, not table-level

Since a single row can genuinely have thousands of columns (the `anchor:`
family on a popular page being the obvious case), BigTable has to be precise
about exactly what "atomic" means here:

- **Updates to a single row are atomic**, no matter how many column families
  or individual columns are touched by that update.
- **There is no support for transactions spanning multiple rows atomically.**
  If an application needs several rows to change together as one unit, that
  guarantee doesn't exist at the BigTable layer — the application has to
  either avoid needing it, or build that guarantee itself on top.

This is a direct, deliberate trade-off against full ACID transactions, in
exchange for the scalability the rest of the design buys.


## 6. Tablets — the unit of distribution

Rows with **consecutive row keys** are grouped together into **Tablets**,
which are the actual unit of distribution and load balancing across servers.
By default, a Tablet is automatically **split once it grows beyond 1 GB**,
keeping individual Tablets — and the servers responsible for them — at a
manageable size as data grows.

The immutable, on-disk data structure BigTable actually uses to store a
Tablet's data is called an **SSTable** — covered in detail next.


## 7. SSTables — the immutable building block everything else depends on

The BigTable paper's own definition: *"An SSTable provides a persistent,
ordered immutable map from keys to values, where both keys and values are
arbitrary byte strings."*

Structurally:

- Inside an SSTable are compressed **64 KB data blocks**, together with an
  **index** used to locate a given row key within those blocks.
- SSTables can also be generated **directly by a MapReduce job** for bulk,
  parallel loading of data — a nice callback to lecture 3's MapReduce material,
  and a genuinely practical technique for ingesting large datasets into
  BigTable without going through the normal write path row by row.
- An optional **Bloom filter** (lecture 7) can be attached to an SSTable to
  speed up checks for row/column pairs that *don't* exist — exactly the
  "skip the expensive lookup when the filter says no" pattern from lecture 7's
  worked example.

### Why immutability is the key that unlocks both of these tricks

This is the single most important design insight in the whole SSTable concept,
and it's worth stating plainly: **immutability is what makes both compression
and Bloom filters practical here.**

- **Compression** in a traditional, mutable-in-place database design is
  genuinely difficult, because a random write into already-compressed data
  generally forces re-compressing everything around it — compression and
  random in-place updates fight each other directly.
- **Bloom filters** are hard to use correctly when items can be *deleted* from
  the underlying data, because (as lecture 7's "negatives" section explained)
  a Bloom filter has no safe delete operation — you can't just unset a bit
  without risking breaking lookups for other items sharing that bit.

An SSTable being immutable sidesteps both problems at once: it's compressed
**once**, when it's written, and never modified afterward — so compression
never has to be redone. And its Bloom filter, similarly, is built once at write
time and never needs an unsupported delete operation, because the SSTable
itself never has anything deleted from it directly (deletion is handled
differently — see the tombstones and compaction discussion below).


## 8. Chubby — the coordination backbone underneath BigTable

BigTable doesn't implement its own leader election or cluster-membership
tracking — it leans entirely on **Chubby**, the fault-tolerant distributed
coordination system (Google's own name for it: a "lock service") introduced in
lecture 6.

The specific jobs BigTable delegates to Chubby, quoted directly from the
BigTable paper: *"...to ensure that there is at most one active master at any
time; to store the bootstrap location of Bigtable data; to discover tablet
servers and finalize tablet server deaths; and to store Bigtable schemas."*

That's leader election, service discovery, failure detection, and metadata
storage — four genuinely different jobs, and all four reduce to the same
underlying consensus problem lecture 6 spent an entire lecture on. The
dependency runs deep enough that the lecture states it plainly: **if the
Chubby service goes down, all other Google services relying on it will
eventually go down too** — a single, critical point that essentially the
entire rest of Google's infrastructure sits on top of, which is precisely why
Chubby itself has to be built on a proven consensus protocol (Paxos) rather
than anything simpler.


## 9. Tablet lookup — finding the right server

A client looking up a piece of data doesn't know in advance which server holds
the relevant Tablet, so BigTable needs a lookup path:

1. Ask **Chubby** for the location of the **root Tablet**.
2. The root Tablet points to an additional layer of index Tablets.
3. That index layer, in turn, points to the actual **user data Tablets** and
   the servers currently responsible for them.

This is a small, bounded hierarchy — not a single flat directory of every
Tablet in the system — which is what lets the lookup stay fast even as the
number of Tablets across the whole cluster grows very large.


### BigTable tablet lookup hierarchy (Chubby → root tablet → index → user data tablets)

![BigTable tablet lookup hierarchy (Chubby → root tablet → index → user data tablets)](images/bigtable_tablet_lookup.png)


## 10. The write path

When a Tablet server receives a write:

1. The write is first **committed to a commit log on GFS** — durability comes
   first, before anything else happens.
2. Once committed, the write is added to an in-memory structure called the
   **memtable**.
3. When the memtable fills up, all the data currently held in it is **sorted
   in memory**, then written out as a new **SSTable on GFS** — and only at
   that point can the commit log file covering that data be safely removed,
   since the data is now durably captured in the SSTable instead.

Notice that this write path is **entirely sequential** at every step: append
to a log, buffer in memory, flush a sorted batch to a new immutable file. There
is no point anywhere in this sequence where BigTable goes back and modifies an
existing on-disk file in place — which is exactly the constraint GFS imposes,
and exactly the constraint that makes the whole design "write optimized."

![BigTable Write Path](images/write_path.png)


In [5]:
# A simplified simulation of the memtable -> SSTable write path, purely to
# make the mechanics concrete. Not a real storage engine -- no actual disk
# I/O, compression, or commit-log durability, just the logical flow.

class ToyBigTable:
    def __init__(self, memtable_flush_threshold=4):
        self.memtable = {}                      # unsorted, in-memory, mutable
        self.sstables = []                       # list of dicts, newest LAST
        self.flush_threshold = memtable_flush_threshold
        self.commit_log = []                     # simulates the GFS commit log

    def write(self, row, column, timestamp, value):
        # Step 1: commit to the (simulated) commit log first
        self.commit_log.append((row, column, timestamp, value))
        # Step 2: add to the memtable
        self.memtable[(row, column, timestamp)] = value
        print(f"WRITE  ({row}, {column}, t={timestamp}) -> memtable "
              f"[{len(self.memtable)}/{self.flush_threshold}]")

        # Step 3: flush if the memtable is full
        if len(self.memtable) >= self.flush_threshold:
            self._flush()

    def _flush(self):
        # Sort in memory, write out as a new (immutable!) SSTable, then the
        # commit log covering this data can be discarded.
        sorted_items = dict(sorted(self.memtable.items()))
        self.sstables.append(sorted_items)
        print(f"  -> FLUSH: memtable written as SSTable #{len(self.sstables)} "
              f"({len(sorted_items)} entries), commit log cleared")
        self.memtable = {}
        self.commit_log = []


tbl = ToyBigTable(memtable_flush_threshold=3)
tbl.write("com.cnn.www", "contents:", 1000, "v1")
tbl.write("com.bbc.www", "contents:", 1000, "v1")
tbl.write("com.cnn.www", "contents:", 2000, "v2")   # triggers a flush (3rd write)
tbl.write("com.yle.fi", "contents:", 1500, "v1")

print(f"\nCurrent SSTables on 'disk': {len(tbl.sstables)}")
print(f"Current memtable (not yet flushed): {tbl.memtable}")


WRITE  (com.cnn.www, contents:, t=1000) -> memtable [1/3]
WRITE  (com.bbc.www, contents:, t=1000) -> memtable [2/3]
WRITE  (com.cnn.www, contents:, t=2000) -> memtable [3/3]
  -> FLUSH: memtable written as SSTable #1 (3 entries), commit log cleared
WRITE  (com.yle.fi, contents:, t=1500) -> memtable [1/3]

Current SSTables on 'disk': 1
Current memtable (not yet flushed): {('com.yle.fi', 'contents:', 1500): 'v1'}


## 11. The read path

When a Tablet server receives a read, the search order matters, and it's the
mirror image of the write path's structure:

1. Check the **memtable** first — the newest data, still only in memory, lives
   there.
2. If not found there, scan the **SSTables**, one at a time, from **newest to
   oldest**, until the SSTable holding the latest version of the requested item
   is found.
3. **Bloom filters** attached to SSTables help here directly: if a filter says
   "definitely not in this SSTable," that whole SSTable can be skipped without
   ever reading its actual data blocks from disk.

The direct consequence of "newest to oldest": the *more* SSTables have
accumulated for a given Tablet, the more of them a read might have to check
before finding (or definitively ruling out) the item being looked up. This is
exactly the problem the next section — compaction — exists to solve.

![BigTable read path](images/read_path.png)


In [8]:
def read(tbl, row, column):
    # Step 1: memtable first
    memtable_matches = [(t, v) for (r, c, t), v in tbl.memtable.items()
                         if r == row and c == column]
    if memtable_matches:
        t, v = max(memtable_matches)
        print(f"READ ({row}, {column}) -> found in memtable at t={t}: {v!r}")
        return v

    # Step 2: scan SSTables newest to oldest
    for i, sstable in enumerate(reversed(tbl.sstables)):
        sstable_index = len(tbl.sstables) - i
        matches = [(t, v) for (r, c, t), v in sstable.items()
                   if r == row and c == column]
        if matches:
            t, v = max(matches)
            print(f"READ ({row}, {column}) -> found in SSTable #{sstable_index} "
                  f"at t={t}: {v!r} (had to check {i + 1} SSTable(s))")
            return v

    print(f"READ ({row}, {column}) -> not found in memtable or any SSTable")
    return None


read(tbl, "com.cnn.www", "contents:")   # this row was flushed to an SSTable
read(tbl, "com.yle.fi", "contents:")    # this one is still sitting in the memtable


READ (com.cnn.www, contents:) -> found in SSTable #1 at t=2000: 'v2' (had to check 1 SSTable(s))
READ (com.yle.fi, contents:) -> found in memtable at t=1500: 'v1'


'v1'

## 12. Compactions — keeping the SSTable count under control

If data keeps getting written continuously, the number of SSTables for a given
Tablet just keeps growing — and since reads scan SSTables newest-to-oldest,
read performance **degrades over time** as that count climbs.

The fix: a background process called **compaction**, which merges several
SSTables into a single new SSTable, using a **merge-sort-like algorithm** —
practical precisely *because* every input SSTable is already internally sorted
by row key, so merging them is a straightforward multi-way merge rather than a
full re-sort from scratch.

### Deletion via tombstones

Because SSTables are immutable, BigTable can't just go delete a data item from
one directly. Instead, deletion is recorded as a special **tombstone**
record — a marker saying "this item has been deleted" — which itself gets
written and stored just like any other update.

A **major compaction** — one that merges *every* SSTable for a Tablet into a
single file — is the only point at which a tombstone record and the data item
it marks as deleted can actually both be **physically removed** from storage
for good. Until a major compaction happens, the tombstone and the "deleted"
data both keep existing somewhere on disk, with the tombstone simply telling
reads to ignore the older version.

### Where this idea comes from

The compaction approach traces back to a specific, foundational paper: Patrick
O'Neil, Edward Cheng, Dieter Gawlick, Elizabeth O'Neil, *The Log-Structured
Merge-Tree (LSM-Tree)*, Acta Informatica 33(4): 351–385 (1996). BigTable's whole
memtable/SSTable/compaction design is essentially a large-scale, distributed
implementation of the LSM-tree idea — and this same underlying idea reappears
in essentially every system discussed later in this lecture.

Compactions only ever need **sequential** reads and writes — which, notably, is
also **the only kind of I/O a WORM filesystem like GFS actually supports** in
the first place, so this isn't an optional design nicety, it's a hard
requirement given the filesystem BigTable is built on. Compactions can be
scheduled during periods of low system activity, since they're background
maintenance work rather than something a client request is waiting on directly.

One practical warning worth keeping in mind: **a large number of deletes can
genuinely hurt BigTable's performance**, precisely because each delete adds
another tombstone that has to be carried around (and checked against, on
reads) until the next major compaction finally clears it out.


In [7]:
# A small simulation of tombstones + major compaction, to make the
# "deletion is deferred until compaction" idea concrete.

TOMBSTONE = object()  # sentinel value marking a deletion

def apply_delete(tbl, row, column, timestamp):
    # A delete is recorded exactly like a write, just with a tombstone value.
    tbl.write(row, column, timestamp, TOMBSTONE)


def major_compaction(tbl):
    """Merge every SSTable (and the current memtable) into one SSTable,
    physically dropping any item whose LATEST version is a tombstone."""
    merged = {}
    # Fold memtable and all SSTables together, keeping only the latest
    # timestamp seen for each (row, column).
    all_sources = list(tbl.sstables) + [tbl.memtable]
    for source in all_sources:
        for (row, column, ts), value in source.items():
            key = (row, column)
            if key not in merged or ts > merged[key][0]:
                merged[key] = (ts, value)

    # Physically drop entries whose latest version is a tombstone
    surviving = {
        (row, column, ts): value
        for (row, column), (ts, value) in merged.items()
        if value is not TOMBSTONE
    }

    print(f"Major compaction: merged {len(tbl.sstables)} SSTable(s) + memtable "
          f"into 1 SSTable, dropping tombstoned items "
          f"({len(merged) - len(surviving)} removed)")

    tbl.sstables = [surviving]
    tbl.memtable = {}


tbl2 = ToyBigTable(memtable_flush_threshold=100)  # avoid auto-flush for this demo
tbl2.write("com.cnn.www", "contents:", 1000, "v1")
tbl2.write("com.bbc.www", "contents:", 1000, "v1")
apply_delete(tbl2, "com.bbc.www", "contents:", 2000)   # bbc entry now tombstoned

print(f"\nBefore compaction, read for bbc.www:")
read(tbl2, "com.bbc.www", "contents:")   # would return the tombstone object itself here

major_compaction(tbl2)

print(f"\nAfter compaction, read for bbc.www:")
result = read(tbl2, "com.bbc.www", "contents:")
print(f"  (result is None: the item and its tombstone are both now gone for good)")


WRITE  (com.cnn.www, contents:, t=1000) -> memtable [1/100]
WRITE  (com.bbc.www, contents:, t=1000) -> memtable [2/100]
WRITE  (com.bbc.www, contents:, t=2000) -> memtable [3/100]

Before compaction, read for bbc.www:
READ (com.bbc.www, contents:) -> found in memtable at t=2000: <object object at 0x00000218F8603930>
Major compaction: merged 0 SSTable(s) + memtable into 1 SSTable, dropping tombstoned items (1 removed)

After compaction, read for bbc.www:
READ (com.bbc.www, contents:) -> not found in memtable or any SSTable
  (result is None: the item and its tombstone are both now gone for good)


### HBase compaction schedule (background merge activity over time)

![HBase compaction schedule (background merge activity over time)](images/hbase_compaction_schedule.png)


## 13. BigTable performance numbers

Two performance facts from the lecture's benchmark plots, both following
directly from everything above:

- **Random reads from disk are roughly an order of magnitude slower than
  random writes**, at large scale — the direct, expected cost of a write-
  optimized design that deliberately never does random writes at all.
- **Table scans are much faster than single-row accesses**, and **random reads
  served from DRAM are dramatically faster than random reads from disk** — both
  entirely consistent with the sequential-vs-random and RAM-vs-disk latency
  gaps quantified back in lecture 5.


### BigTable performance benchmark plots (random read/write, scan speed)

![BigTable performance benchmark plots (random read/write, scan speed)](images/bigtable_performance.png)


## 14. BigTable — pulling the design together

- BigTable is a **scalable, write-optimized** database design.
- It uses **only sequential writes** for improved write performance — the one
  constraint everything else in this lecture flows from.
- For **read-intensive workloads where the working set mostly fits in DRAM**,
  BigTable can be an excellent fit.
- For **read-intensive workloads much larger than available DRAM**, a
  traditional RDBMS is still likely to be the better match — this is a genuine
  limitation, not a minor caveat.
- BigTable's **scan speed is impressive**, which makes it a particularly good
  fit for scan-heavy workloads — the lecture's example being MapReduce jobs
  reading BigTable as their input source.
- The aggressive **compression and Bloom filter** optimizations that make all
  of this work are only viable **because SSTables are immutable** — this point
  from earlier is worth restating as the lecture's own closing summary of the
  whole design.


## 15. Apache HBase — the open-source clone

**Apache HBase** follows BigTable's design very closely, with a small number
of concrete substitutions:

| BigTable component | HBase equivalent |
|---|---|
| Google File System (GFS) | HDFS |
| Chubby | Apache ZooKeeper |
| SSTable | HFile (and HFile V2) |

One further difference worth noting: HBase only **partially** supports fully
memory-mapped data, unlike BigTable's original design.

Every substitution here should feel familiar — this is a direct instance of the
general pattern from lectures 3, 4, and 6: HDFS as the open filesystem
replacing GFS, and ZooKeeper as the open coordination service replacing Chubby.
HBase isn't a reinvention of BigTable's ideas; it's the same architecture,
rebuilt entirely on the open-source equivalents of Google's internal
infrastructure.


## 16. HBase in production — Facebook Messaging

HBase sees real production use at companies including **AirBnB**, **Imgur**,
and **Pinterest**. The lecture's detailed worked example, though, comes from
Facebook: Borthakur et al., *Apache Hadoop Goes Realtime at Facebook*, SIGMOD
Conference 2011: 1071–1080 — describing the infrastructure behind Facebook
Messaging at the scale it operated at:

- **500 million users**
- **Several petabytes** of data
- **135,000,000,000 messages per month** — that's 135 billion messages a
  month, or roughly 52,000 messages every single second, sustained.

### Why Facebook picked HBase for this

- **Elasticity** — the ability to add new storage nodes as needed, without a
  disruptive redesign.
- **High write throughput** — exactly what a write-optimized, LSM-tree-based
  design is built for.
- **Consistency guarantees within a single datacenter.**
- **Comparable — though not better — read performance** relative to a
  traditional RDBMS; HBase wasn't chosen because it read faster, it was chosen
  despite reading no better than the alternative, because everything else on
  this list mattered more for this workload.
- **High availability and disaster recovery features.**
- **Fault isolation** — worth pausing on the specific number the lecture gives
  here: at Facebook's scale, **a hard disk fails roughly every 30 minutes**
  somewhere in the fleet. That's not a hypothetical edge case to design around
  eventually — it's routine, ongoing background noise the system has to
  isolate cleanly, every single day.
- **Atomic read-modify-write primitives** — for example, implementing atomic
  counters, which fits neatly with lecture 8's earlier point about BigTable's
  row-level atomicity guarantee.
- **Efficient range scans** — the lecture's example: fetching the last 100
  messages for a given user, which is exactly the kind of "consecutive row
  keys, sorted, scan forward" access pattern BigTable's Tablet layout was built
  to make cheap.


### Typical 100-node HBase deployment (Facebook Messages architecture)

![Typical 100-node HBase deployment (Facebook Messages architecture)](images/hbase_100node_deployment.png)


## 17. Challenges Facebook hit along the way

- **Each user is served by a single cluster**, but Facebook operates *many*
  separate 100-node HBase clusters in parallel — the sharding happens at the
  cluster level, not within one giant shared cluster.
- **HDFS was originally designed for batch processing, not realtime** —
  carrying assumptions like large timeouts that are a poor fit for a
  latency-sensitive, interactive messaging workload, and had to be adapted
  accordingly.
- **The HDFS NameNode single point of failure** (a limitation flagged back in
  lecture 3–4's HDFS discussion) had to be directly addressed for this use
  case. Facebook's solution was **AvatarNode** — a hot-backup NameNode design
  that keeps a standby NameNode ready to take over quickly, rather than leaving
  a NameNode failure as a full outage.


## 18. Beyond BigTable and HBase — the same idea, everywhere

The lecture closes by tracing the LSM-tree/SSTable idea forward into a whole
family of systems still in wide use today — worth recognizing as the same core
technique the rest of this lecture just walked through in detail, not a
separate concept:

- **LevelDB** (<https://github.com/google/leveldb>) — a lightweight database
  engine, written by the original BigTable authors themselves, using similar
  design trade-offs at a much smaller scale. Notably, it's used inside
  **Google Chrome**.
- **RocksDB** (<http://rocksdb.org/>) — a key-value store developed further
  from LevelDB, adding features and performance tuning for a broader range of
  workloads.
- **MyRocks** (<http://myrocks.io/>) — MySQL, running with RocksDB as its
  storage backend instead of the traditional InnoDB.
- **CockroachDB** (<https://www.cockroachlabs.com/>) — a distributed database
  built on top of RocksDB (and previously introduced back in lecture 7 as one
  of the open-source systems inspired by Google Spanner's design goals).
- **Apache Cassandra** (<http://cassandra.apache.org/>) — also uses log-
  structured merge trees to implement its own storage layer, tying it back
  into the key-value-store family from lecture 7's taxonomy.

### Facebook's own migration — a real-world data point

In 2018, Facebook migrated a significant piece of its own infrastructure from
**MySQL (with the InnoDB backend) and HBase**, over to **MyRocks** (MySQL
running on the RocksDB backend). Details:
<https://www.slideshare.net/MariaDB/migrating-from-innodb-and-hbase-to-myrocks-at-facebook>

The lecture draws one specific, important conclusion from this migration: the
**performance difference observed between MyRocks and HBase came down to
implementation efficiency, not database architecture.** Both systems are, at
their core, implementing the same LSM-tree idea this entire lecture has been
building toward — the gains Facebook saw weren't from switching to a
fundamentally different design, they came from a better-engineered
implementation of the *same* design.


## Summary

This lecture is really one idea, examined from every angle: **if a filesystem
only supports sequential writes, build the entire database around never doing
anything else.** BigTable's memtable, its immutable SSTables, its
newest-to-oldest read path, and its background compactions are all direct
consequences of that one constraint — and once you see it that way, the row-
level-only atomicity guarantee, the reliance on Chubby/ZooKeeper for
coordination, and the read/write performance asymmetry in the benchmark plots
all stop looking like a list of separate design choices and start looking like
the inevitable shape a database takes once "sequential writes only" is
accepted as a non-negotiable starting constraint. That single idea —
BigTable's specific implementation of the more general **log-structured
merge-tree** — turned out to be valuable enough to outlive BigTable itself,
resurfacing in LevelDB, RocksDB, MyRocks, CockroachDB, and Cassandra, right up
to Facebook's own 2018 decision to move off HBase and onto a different, more
efficient implementation of the very same underlying idea.
